In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon
import itertools
import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))
import liset_data_reader.lists_sessions as lists_sessions
# Set style
sns.set_style("whitegrid")
# plt.rcParams.update({'font.size': 12})
plt.rcParams['svg.fonttype'] = 'none'

In [2]:
# Load Data
results_path = os.path.join("spikes", "all_networks_metrics_buzsaki.csv")

if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} rows from {results_path}")
    
    # Preprocessing
    # Rename ADAPT to Adapt for consistency
    if "ADAPT" in df.columns:
        df.rename(columns={"ADAPT": "Adapt"}, inplace=True)
        
    print("Columns:", df.columns.tolist())
    print("Threshold:", sorted(df["Threshold"].unique()))
    print("Fall_off:", sorted(df["Fall_off"].unique()))
    
    display(df.head())
else:
    print(f"File not found: {results_path}. Please run process_live_spikes.py first.")

Loaded 2400 rows from spikes\all_networks_metrics_buzsaki.csv
Columns: ['Session', 'Channel', 'Threshold', 'Fall_off', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1', 'Mean_Latency', 'Num_Spikes', 'Num_Ripples', 'Duration_min']
Threshold: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
Fall_off: [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0]


,Session,Channel,Threshold,Fall_off,TP,FP,FN,Precision,Recall,F1,Mean_Latency,Num_Spikes,Num_Ripples,Duration_min
0,PV01ai32FPGA_250611_122326,20,1.0,0.5,138,2130,31,0.060847,0.816568,0.113254,0.854589,3164,169,11.670613
1,Calb_251210_115904,15,1.0,0.5,44,2108,12,0.020446,0.785714,0.039855,6.588636,3011,56,11.571627
2,Calbai32FPGA_251003_144832,11,1.0,0.5,160,1701,33,0.085975,0.829016,0.155794,5.025000,2515,193,11.366827
3,Calb_251210_121141,15,1.0,0.5,56,1899,13,0.028645,0.811594,0.055336,1.580357,2771,69,10.848569
4,Calb_251210_122327,13,1.0,0.5,73,1852,18,0.037922,0.802198,0.072421,1.526941,2708,91,10.954382


In [4]:
hfo_sessions = lists_sessions.HFO_sessions
ripple_sessions = lists_sessions.Ripple_sessions

sessions_to_reject={
    # "2025-09-24_16-29-07", #R   # Nothing concrete to reject here
        "2025-09-24_17-38-17", # Barely any ripples (14 in total...)
        "2025-09-25_12-52-22", # Not a good session to detect ripples
        #  "2025-09-24_11-34-51",
        } #R 
# og_session_set={"2025-09-22_17-55-26", #R
# #                 "2025-09-23_15-50-26", #R
# #                 "2025-09-24_10-24-40", #R
# #                 "2025-09-24_14-22-55", #H
# #                 "2025-09-24_15-13-10", #H
# #                 "2025-09-25_16-41-14"} #R    
# }
sessions_to_reject.update(hfo_sessions)

    # df=df[df["Session"].isin(og_session_set)]

    # # New sessions:

    

df=df[~df["Session"].isin(sessions_to_reject)]
print(df["Session"].unique(),len(df["Session"].unique()))
# df=df[~df["Session"].isin(ripple_sessions)]

['PV01ai32FPGA_250611_122326' 'Calb_251210_115904'
 'Calbai32FPGA_251003_144832' 'Calb_251210_121141' 'Calb_251210_122327'
 'Calb_251211_104316' 'Calb_251210_164150' 'Calb_251210_165332'
 'Calb_251211_110650' 'Calb_251211_105518' 'Calb_251210_162849'
 'Calb_251209_160255' 'PV01ai32FPGA_250611_115923'
 'Calbai32FPGA_251003_150055' '2025-09-23_16-17-52' '2025-09-23_15-50-26'
 '2025-09-22_17-42-27' '2025-09-24_11-34-51' '2025-09-24_16-29-07'
 '2025-09-25_11-21-53' '2025-09-22_17-55-26' '2025-09-24_10-24-40'
 '2025-09-25_16-41-14'] 23


In [6]:
# Print summary
print("\n--- Summary by Network ---")
summary = df.groupby([ "Threshold","Fall_off"])[["F1", "Precision", "Recall", "TP", "FP", "FN"]].median().round(3).sort_values(by="F1", ascending=False)
display(summary)


--- Summary by Network ---


F1  Precision  Recall    TP      FP    FN
Threshold Fall_off                                              
5.0       4.5       0.711      0.710   0.676  79.0    29.0  45.0
6.0       4.0       0.701      0.655   0.700  82.0    33.0  42.0
          5.0       0.696      0.760   0.624  72.0    16.0  55.0
          4.5       0.696      0.723   0.659  76.0    25.0  48.0
7.0       4.5       0.691      0.741   0.613  68.0    17.0  53.0
...                   ...        ...     ...   ...     ...   ...
3.0       0.5       0.316      0.191   0.919  95.0   436.0   8.0
2.0       1.5       0.262      0.152   0.938  97.0   588.0   7.0
          1.0       0.192      0.107   0.924  95.0   853.0   9.0
          0.5       0.171      0.095   0.890  97.0   949.0  11.0
1.0       0.5       0.097      0.051   0.809  91.0  1757.0  19.0

[96 rows x 6 columns]